# MinHash Statistical Properties

## Learning Objectives
- Derive the variance and confidence intervals for the MinHash Jaccard estimator
- Determine the number of hash functions needed for a target accuracy
- Prove that the estimator is unbiased and verify empirically
- Contrast MinHash with naive random sampling
- Combine multiple MinHash sketches correctly

## Recap: The MinHash Estimator

For a single hash function h, define the indicator:

$$X_i = \mathbf{1}[\text{sig}(A)[i] = \text{sig}(B)[i]]$$

The MinHash theorem (proved in `ml_002_01_minhash_algorithm.ipynb`) states:

$$P[X_i = 1] = J(A, B)$$

So each $X_i$ is **Bernoulli(J)**. The estimator using k hash functions is:

$$\hat{J} = \frac{1}{k} \sum_{i=1}^{k} X_i$$

This is a sample mean of i.i.d. Bernoulli(J) random variables.

## Variance Derivation

Since $X_i \sim \text{Bernoulli}(J)$:

$$\text{Var}(X_i) = J(1 - J)$$

Since the $k$ hash functions are independent:

$$\text{Var}(\hat{J}) = \text{Var}\!\left(\frac{1}{k} \sum_{i=1}^k X_i\right) = \frac{1}{k^2} \sum_{i=1}^k \text{Var}(X_i) = \frac{J(1-J)}{k}$$

**Upper bound**: the function $J(1-J)$ is maximized at $J = 0.5$, giving:

$$\text{Var}(\hat{J}) \leq \frac{1}{4k}$$

**Standard deviation bound**:

$$\text{std}(\hat{J}) = \sqrt{\frac{J(1-J)}{k}} \leq \frac{1}{2\sqrt{k}}$$

This matches the bound quoted in the algorithm notebook. The variance is worst when the true similarity is 0.5 — exactly the threshold region where we care most about accuracy.

## Confidence Intervals

By the Central Limit Theorem (justified for $k \geq 30$), $\hat{J}$ is approximately normal:

$$\hat{J} \approx \mathcal{N}\!\left(J,\ \frac{J(1-J)}{k}\right)$$

A 95% confidence interval is:

$$\hat{J} \pm 1.96 \cdot \sqrt{\frac{J(1-J)}{k}}$$

Since the true $J$ is unknown, substitute the worst-case $J=0.5$:

$$\hat{J} \pm \frac{0.98}{\sqrt{k}}$$

| k | +/- half-width (95% CI) |
|---|-----------------------|
| 50 | +/- 0.139 |
| 100 | +/- 0.098 |
| 200 | +/- 0.069 |
| 400 | +/- 0.049 |
| 1000 | +/- 0.031 |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

def empirical_ci(j_true, k, n_trials=2000, rng=None):
    if rng is None:
        rng = np.random.default_rng(0)
    estimates = (rng.random((n_trials, k)) < j_true).mean(axis=1)
    return estimates.mean(), estimates.std(), np.percentile(estimates, [2.5, 97.5])

# J=0.5 is the worst case for variance
j_true = 0.5
print(f'=== Empirical CI verification (J={j_true}, 2000 trials) ===')
for k in [50, 100, 200, 400, 1000]:
    j_hat, std_emp, (lo, hi) = empirical_ci(j_true, k, rng=rng)
    theoretical_half = 0.98 / np.sqrt(k)
    print(f'  k={k:5d}: j_hat={j_hat:.4f}  std={std_emp:.4f}  '
          f'95%CI=[{lo:.4f},{hi:.4f}]  theory +/-{theoretical_half:.4f}')

In [ ]:
# plot: empirical std vs k, overlaid with theoretical bound
ks = np.arange(10, 1001, 10)
theoretical_std = 1 / (2 * np.sqrt(ks))

empirical_stds = []
for k in ks:
    _, s, _ = empirical_ci(0.5, int(k), n_trials=500, rng=rng)
    empirical_stds.append(s)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ks, theoretical_std, 'b-', label='Theoretical bound 1/(2*sqrt(k))')
ax.plot(ks, empirical_stds, 'r--', alpha=0.8, label='Empirical std (J=0.5, 500 trials)')
k200_theory = 1 / (2 * np.sqrt(200))
ax.axvline(200, color='gray', linestyle=':', lw=1)
ax.axhline(k200_theory, color='gray', linestyle=':', lw=1)
ax.annotate(f'k=200 -> +/-{0.98/np.sqrt(200):.3f}', xy=(200, k200_theory),
            xytext=(350, 0.065), arrowprops=dict(arrowstyle='->'))
ax.set_xlabel('Number of hash functions (k)')
ax.set_ylabel('Std of J estimate')
ax.set_title('MinHash accuracy vs number of hash functions')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('minhash_accuracy_vs_k.png', dpi=100)
plt.show()
print('saved minhash_accuracy_vs_k.png')
print(f'At k=200: theoretical +/-CI = +/-{0.98/np.sqrt(200):.3f}, '
      f'memory = {200*8} bytes per doc (int64)')

## Bias Analysis

**Claim**: $\hat{J}$ is an **unbiased** estimator of $J$.

**Proof**:

$$E[\hat{J}] = E\!\left[\frac{1}{k} \sum_{i=1}^k X_i\right] = \frac{1}{k} \sum_{i=1}^k E[X_i] = \frac{1}{k} \cdot k \cdot J = J$$

The estimator is unbiased for *all* values of k, including k=1. Small k is noisy but not systematically off. This is a desirable property — you can always improve accuracy by adding more hash functions without introducing bias.

Compare with truncated estimators or estimators that clip values to [0,1]: those can introduce bias at the boundaries.

In [ ]:
# empirical bias check: estimated J vs true J across a range of true J values
j_values = np.linspace(0.05, 0.95, 18)

fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=False)
for ax, k in zip(axes, [10, 100, 500]):
    means_list = []
    stds_list  = []
    for j_t in j_values:
        trials = (rng.random((300, k)) < j_t).mean(axis=1)
        means_list.append(trials.mean())
        stds_list.append(trials.std())
    ax.errorbar(j_values, means_list, yerr=[1.96*s for s in stds_list],
                fmt='o', capsize=3, label=f'k={k}')
    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Ideal (unbiased)')
    ax.set_xlabel('True J')
    ax.set_title(f'k = {k}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel('Estimated J (mean +/- 95% CI)')
fig.suptitle('MinHash Bias Check: Estimated vs True Jaccard', fontsize=13)
plt.tight_layout()
plt.savefig('minhash_bias_check.png', dpi=100)
plt.show()
print('saved minhash_bias_check.png')
print('All estimates fall on the diagonal -> estimator is unbiased at all k')

## Comparison: MinHash vs Random Sampling

An alternative approach: randomly sample m elements from A union B and estimate J by the fraction that fall in A intersect B.

Why MinHash is better:

| Property | MinHash | Random Sampling |
|---|---|---|
| Requires knowing A union B in advance? | No | Yes (or two passes) |
| Handles sparse sets? | Yes | Fails (rarely samples intersection) |
| Handles very different set sizes? | Yes | Biased toward larger set |
| Streaming compatible? | Yes | No (needs full set) |
| Per-document storage | k integers | m integers (similar) |

MinHash is particularly powerful for *streaming* settings: as new shingles arrive, maintain the running minimum of each hash function. The signature can be updated in O(k) per shingle without storing the full shingle set.

## Combining Multiple MinHash Estimates

Suppose you computed two independent MinHash sketches for the same document collection: sketch 1 with $k_1$ hash functions, sketch 2 with $k_2$ functions.

**Question**: how should you combine the two estimates?

**Answer**: treat all $k_1 + k_2$ hash function agreements as a single sample mean.

The combined estimator $\hat{J}_{\text{comb}} = \frac{k_1 \hat{J}_1 + k_2 \hat{J}_2}{k_1 + k_2}$ has:

$$\text{std}(\hat{J}_{\text{comb}}) \leq \frac{1}{2\sqrt{k_1 + k_2}}$$

This is better than either sketch alone. The two sketches must use *different* (independent) hash functions — using the same functions twice does not reduce variance.

In [ ]:
# verify combined estimate variance
j_true = 0.5
n_trials = 3000

configs = [
    (100,   0, 'sketch1 only (k=100)'),
    (  0, 100, 'sketch2 only (k=100)'),
    ( 50,  50, 'combined (k1=50, k2=50)'),
    (100, 100, 'combined (k1=100, k2=100)'),
]

print(f'True J = {j_true},  {n_trials} trials')
print(f'{"Configuration":<35}  {"mean":>6}  {"std":>6}  {"theory bound":>13}')
for k1, k2, label in configs:
    if k1 == 0:
        ests = (rng.random((n_trials, k2)) < j_true).mean(axis=1)
    elif k2 == 0:
        ests = (rng.random((n_trials, k1)) < j_true).mean(axis=1)
    else:
        agree1 = (rng.random((n_trials, k1)) < j_true).mean(axis=1)
        agree2 = (rng.random((n_trials, k2)) < j_true).mean(axis=1)
        ests   = (k1 * agree1 + k2 * agree2) / (k1 + k2)
    k_total = k1 + k2
    theory  = 1 / (2 * np.sqrt(k_total))
    print(f'  {label:<33}  {ests.mean():6.4f}  {ests.std():6.4f}  <= {theory:.4f}')